# nb00 - Extract and Profile: CMS Medicare Advantage and Part D Star Ratings

**Run this locally in Jupyter.** It downloads the CMS Star Ratings ZIPs, unzips them into `../data/raw/`, profiles every table, and writes `../data/extraction_log.json`.

After it runs, **paste the profile output (the last long cell) back to Claude** so the cleaning notebook (nb01) can be written to the real structure.

Source: [CMS Part C and D Performance Data](https://www.cms.gov/medicare/health-drug-plans/part-c-d-performance-data). The download is run from your machine, not from Claude's sandbox.

In [1]:
# One-time installs if needed:
# pip install requests pandas openpyxl xlrd
import os, json, zipfile
from pathlib import Path
from datetime import datetime
import requests
import pandas as pd

DATA = Path('..') / 'data'
RAW = DATA / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

# CMS Star Ratings ZIPs. 2024+ use the "data tables" format; 2019-2023 bundle star + display measures.
ALL_URLS = {
    '2026': 'https://www.cms.gov/files/zip/2026-star-ratings-data-tables.zip',
    '2025': 'https://www.cms.gov/files/zip/2025-star-ratings-data-tables.zip',
    '2024': 'https://www.cms.gov/files/zip/2024-star-ratings-data-tables-jul-2-2024.zip',
    '2023': 'https://www.cms.gov/files/zip/2023-star-ratings-and-display-measures.zip',
    '2022': 'https://www.cms.gov/files/zip/2022-star-ratings-and-display-measures.zip',
    '2021': 'https://www.cms.gov/files/zip/2021-star-ratings-and-display-measures.zip',
    '2020': 'https://www.cms.gov/files/zip/2020-star-ratings-and-display-measures.zip',
    '2019': 'https://www.cms.gov/files/zip/2019-star-ratings-and-display-measures.zip',
}

# Start with the 3 recent, consistent "data tables" years. Add older years here once the
# format is understood (they bundle display measures and have different internal layouts).
YEARS_TO_PULL = ['2024', '2025', '2026']

HEADERS = {'User-Agent': 'Mozilla/5.0 (Tableau exam prep research)'}

In [2]:
# Download the ZIPs (skips any already saved)
log = {'pulled_at': datetime.now().isoformat(timespec='seconds'),
       'source': 'CMS Part C and D Performance Data',
       'source_url': 'https://www.cms.gov/medicare/health-drug-plans/part-c-d-performance-data',
       'files': []}

for yr in YEARS_TO_PULL:
    url = ALL_URLS[yr]
    dest = RAW / f'star_ratings_{yr}.zip'
    if dest.exists():
        print('exists, skipping', dest.name)
    else:
        print('downloading', yr, '...')
        r = requests.get(url, headers=HEADERS, timeout=180)
        r.raise_for_status()
        dest.write_bytes(r.content)
        print(f'  saved {dest.name}  {len(r.content)/1e6:.1f} MB')
    log['files'].append({'year': yr, 'url': url, 'zip': dest.name, 'bytes': dest.stat().st_size})

downloading 2024 ...
  saved star_ratings_2024.zip  2.6 MB
downloading 2025 ...
  saved star_ratings_2025.zip  2.4 MB
downloading 2026 ...
  saved star_ratings_2026.zip  2.1 MB


In [3]:
# Unzip each year into ../data/raw/<year>/
for yr in YEARS_TO_PULL:
    zp = RAW / f'star_ratings_{yr}.zip'
    out = RAW / yr
    out.mkdir(exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(out)
    files = [str(p.relative_to(out)) for p in out.rglob('*') if p.is_file()]
    print(f'{yr}: {len(files)} files')
    for f in files:
        print('   ', f)

2024: 14 files
    2024 Star Ratings Data Table - High Performing Contracts (Jul 2 2024).csv
    2024 Star Ratings Data Table - Part D Cut Points (Jul 2 2024).csv
    2024 Star Ratings Data Table - Domain Stars (Jul 2 2024).csv
    2024 Star Ratings Data Table - Notes (Jul 2 2024).csv
    PQA_Medication_List_SUPD_YOS2022_Mar_2023.xlsx
    2024 Star Ratings Data Table - Summary Rating (Jul 2 2024).csv
    2024 Star Ratings Data Table - Measure Data (Jul 2 2024).csv
    PQA_Medication_List_ADH_YOS2022_Mar_2023.xlsx
    2024 Star Ratings Data Table - Measure Stars (Jul 2 2024).csv
    2024 Star Ratings Data Table - Low Performing Contracts (Jul 2 2024).csv
    2024 Star Ratings Data Table - Part C Cut Points (Jul 2 2024).csv
    2024 Star Ratings Data Table - CAI (Jul 2 2024).csv
    2024_Report_Card_Master_Table_2024_07_02.xlsx
    2024 Star Ratings Data Table - Disenrollment Reasons (Jul 2 2024).csv
2025: 13 files
    PQA_Value_Sets_ADH_YOS2023_Mar_2024.xlsx
    2025 Star Ratings Data T

In [4]:
# Profile every tabular file: sheets, shapes, columns, and a peek at the first rows
def profile_file(path):
    print('\n' + '=' * 90)
    print('FILE:', path.relative_to(RAW))
    ext = path.suffix.lower()
    try:
        if ext in ('.xlsx', '.xls'):
            xl = pd.ExcelFile(path)
            print(' sheets:', xl.sheet_names)
            for s in xl.sheet_names:
                df = xl.parse(s, header=None, nrows=6, dtype=str)
                print(f'  -- sheet {s!r}: {df.shape[1]} columns; first rows (header row may be offset):')
                print(df.head(6).to_string(max_colwidth=28))
        elif ext in ('.csv', '.txt'):
            df = pd.read_csv(path, nrows=6, dtype=str, encoding='latin-1')
            print(' cols:', list(df.columns)[:20])
            print(df.head(4).to_string(max_colwidth=28))
        else:
            print(' (skipped, not tabular)')
    except Exception as e:
        print(' ERROR reading:', type(e).__name__, e)

for yr in YEARS_TO_PULL:
    for p in sorted((RAW / yr).rglob('*')):
        if p.is_file() and p.suffix.lower() in ('.xlsx', '.xls', '.csv', '.txt'):
            profile_file(p)


FILE: 2024/2024 Star Ratings Data Table - CAI (Jul 2 2024).csv
 cols: ['2024 CAI View: Medicare Report Card Master Table', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19']
  2024 CAI View: Medicare Report Card Master Table                   Unnamed: 1                   Unnamed: 2                   Unnamed: 3        Unnamed: 4  Unnamed: 5        Unnamed: 6      Unnamed: 7   Unnamed: 8 Unnamed: 9 Unnamed: 10 Unnamed: 11 Unnamed: 12 Unnamed: 13 Unnamed: 14 Unnamed: 15 Unnamed: 16 Unnamed: 17 Unnamed: 18 Unnamed: 19 Unnamed: 20 Unnamed: 21 Unnamed: 22 Unnamed: 23 Unnamed: 24 Unnamed: 25 Unnamed: 26 Unnamed: 27 Unnamed: 28 Unnamed: 29 Unnamed: 30 Unnamed: 31 Unnamed: 32 Unnamed: 33 Unnamed: 34 Unnamed: 35 Unnamed: 36 Unnamed: 37 Unnamed: 38 Unnamed: 39 Unnamed: 40 Un

In [5]:
# Write the extraction log
(DATA / 'extraction_log.json').write_text(json.dumps(log, indent=2))
print('wrote', DATA / 'extraction_log.json')
print(json.dumps(log, indent=2))

wrote ../data/extraction_log.json
{
  "pulled_at": "2026-07-10T18:23:17",
  "source": "CMS Part C and D Performance Data",
  "source_url": "https://www.cms.gov/medicare/health-drug-plans/part-c-d-performance-data",
  "files": [
    {
      "year": "2024",
      "url": "https://www.cms.gov/files/zip/2024-star-ratings-data-tables-jul-2-2024.zip",
      "zip": "star_ratings_2024.zip",
      "bytes": 2562553
    },
    {
      "year": "2025",
      "url": "https://www.cms.gov/files/zip/2025-star-ratings-data-tables.zip",
      "zip": "star_ratings_2025.zip",
      "bytes": 2449297
    },
    {
      "year": "2026",
      "url": "https://www.cms.gov/files/zip/2026-star-ratings-data-tables.zip",
      "zip": "star_ratings_2026.zip",
      "bytes": 2120939
    }
  ]
}
